# Consensus — Career Strategy Agent
### Colab Setup: Ollama + Dependencies + Streamlit UI
Run each cell in order.

In [1]:
#Ollama's installer unpacks with zstd
!apt-get update -qq && apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 60 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (565 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
# ── 1. Install system dependencies ──────────────────────────────────────────
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# ──  Confirm Ollama is alive before proceeding ────────────────────────────
import requests, time

for attempt in range(3):
    try:
        r = requests.get('http://localhost:11434', timeout=5)
        print('✅ Ollama server is up.')
        break
    except Exception:
        print(f'Waiting for Ollama... attempt {attempt+1}/3')
        time.sleep(3)
else:
    print('❌ Ollama not responding. Re-run cells 1 and 2.')


Waiting for Ollama... attempt 1/3
Waiting for Ollama... attempt 2/3
Waiting for Ollama... attempt 3/3
❌ Ollama not responding. Re-run cells 1 and 2.


In [4]:
# ── 2. Start Ollama server in the background ─────────────────────────────────
import subprocess, time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)  # give server time to start
print('Ollama server started.')

Ollama server started.


In [5]:
# Pull tinyllama — 637MB, runs on Colab free CPU in ~2 min
!ollama pull tinyllama
print('Model ready.')


Model ready.


In [6]:
# ── 4. Install Python dependencies ───────────────────────────────────────────
!pip install -q \
    streamlit \
    pymupdf \
    python-docx \
    beautifulsoup4 \
    requests \
    pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 149.2 MB/s eta 0:00:00


In [7]:
# ── 5. Write project files to Colab filesystem ───────────────────────────────
import os
os.makedirs('/content/consensus', exist_ok=True)
print('Project directory created at /content/consensus')

Project directory created at /content/consensus


In [8]:
# ── 6. Verify Ollama is responding ───────────────────────────────────────────
import requests

try:
    r = requests.post(
        'http://localhost:11434/api/generate',
        json={'model': 'tinyllama', 'prompt': 'Say OK', 'stream': False}
    )
    print('Ollama responding:', r.json()['response'].strip())
except Exception as e:
    print('ERROR — Ollama not responding:', e)

Ollama responding: Sure, I'd be happy to help. Just type "OK" or "Yes" at the prompt. Alternatively, you can also use "Hey AI," "Okay," or even "Good job!" depending on your personal preferences. Have a great day!


## Write source files
The next cells write `tools.py`, `agent.py`, and `app.py` into `/content/consensus/`.

In [19]:
%%writefile /content/consensus/tools.py

"""
tools.py — Consensus Career Strategy Agent
All tool functions called by the agent loop.

Tools:
    1. parse_resume(pdf_bytes)         → dict of skills, roles, experience
    2. fetch_job_requirements(url, role) → list of required skills
    3. analyze_gap(resume_data, job_skills) → dict of matched / missing skills
    4. rewrite_resume(resume_data, job_skills, target_role) → str (rewritten resume)
    5. generate_docx(missing_skills, target_role) → bytes (Word document)
"""

import re
import json
import requests
from io import BytesIO
from bs4 import BeautifulSoup
import fitz  # PyMuPDF
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH


# ── LLM helper ───────────────────────────────────────────────────────────────

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL      = "tinyllama"   # ~637MB — much faster on Colab CPU than llama3.2:3b

def _llm(prompt: str, expect_json: bool = False, max_tokens: int = 512) -> str:
    """
    Send a prompt to the local Ollama model and return the response string.
    If expect_json=True, appends a reminder to respond only with valid JSON.
    Raises RuntimeError with a clear message if Ollama is unreachable or times out.
    """
    if expect_json:
        prompt += "\n\nRespond with valid JSON only. No explanation, no markdown fences."

    payload = {
        "model":  MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.1,
            "num_predict": max_tokens,
            "num_ctx": 2048,
        }
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=300)
        response.raise_for_status()
    except requests.exceptions.ConnectionError:
        raise RuntimeError(
            "Cannot connect to Ollama at localhost:11434. "
            "Make sure the Ollama server is running (cell 2 of the notebook)."
        )
    except requests.exceptions.Timeout:
        raise RuntimeError(
            f"Ollama timed out after 300s. The model is too slow on this runtime. "
            "Run: !ollama pull tinyllama  in a notebook cell and restart."
        )
    return response.json()["response"].strip()


def _parse_json_response(raw: str) -> dict | list:
    """
    Safely extract JSON from an LLM response.
    Strips markdown fences if present before parsing.
    """
    clean = re.sub(r"```(?:json)?", "", raw).replace("```", "").strip()
    return json.loads(clean)


# ── Tool 1: Resume Parser ─────────────────────────────────────────────────────

def parse_resume(pdf_bytes: bytes) -> dict:
    """
    Tool 1 — Resume Parser.

    Accepts raw PDF bytes, extracts text, then uses the LLM to return
    a structured JSON breakdown of the candidate's profile.

    Returns:
        {
            "name": str,
            "skills": [str, ...],
            "roles": [{"title": str, "company": str, "years": float}, ...],
            "education": [str, ...],
            "summary": str
        }
    """
    # Extract raw text from PDF
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    raw_text = ""
    for page in doc:
        raw_text += page.get_text()
    doc.close()

    # ── Step 1: Heuristic extraction (fast, reliable, no LLM needed) ──────────
    result = _heuristic_parse(raw_text)

    # ── Step 2: Try LLM to enrich — if it fails, heuristic result still works ──
    prompt = f"""Extract skills and job titles from this resume. Return JSON only.
Format: {{"name": "...", "skills": ["skill1","skill2"], "roles": [{{"title":"...","company":"...","years":1}}], "education": ["..."], "summary": "..."}}
Resume (first 1500 chars):
{raw_text[:1500]}"""
    try:
        raw = _llm(prompt, expect_json=True, max_tokens=400)
        parsed = _parse_json_response(raw)
        # Only use LLM result if it returned a proper dict with skills
        if isinstance(parsed, dict) and isinstance(parsed.get("skills"), list) and len(parsed["skills"]) > 0:
            # Merge: keep heuristic name if LLM returns Unknown
            if parsed.get("name", "Unknown") in ("Unknown", "", None):
                parsed["name"] = result["name"]
            # Combine skills from both sources
            all_skills = list(set(result["skills"] + parsed.get("skills", [])))
            parsed["skills"] = all_skills
            return parsed
    except Exception:
        pass  # LLM failed — heuristic result is good enough

    return result


def _heuristic_parse(raw_text: str) -> dict:
    """
    Rule-based resume parser. Extracts skills, name, and roles using
    keyword matching and common resume patterns. Used as primary parser
    and as fallback when the LLM returns bad JSON.
    """
    lines = [l.strip() for l in raw_text.split("\n") if l.strip()]

    # ── Name: usually the first non-empty line ────────────────────────────────
    name = lines[0] if lines else "Unknown"
    # Reject lines that look like section headers or email addresses
    if any(kw in name.lower() for kw in ["resume", "cv", "curriculum", "@", "http"]):
        name = lines[1] if len(lines) > 1 else "Unknown"

    # ── Skills: keyword match against a broad tech/soft-skills vocabulary ─────
    SKILL_KEYWORDS = [
        # Languages & core tech
        "python","java","javascript","typescript","c++","c#","ruby","go","rust","swift",
        "kotlin","r","matlab","scala","php","sql","nosql","bash","shell","html","css",
        # ML / Data
        "machine learning","deep learning","nlp","computer vision","tensorflow","pytorch",
        "keras","scikit-learn","pandas","numpy","matplotlib","seaborn","hugging face",
        "llm","transformers","rag","fine-tuning","data analysis","data science",
        "data engineering","feature engineering","statistics","a/b testing",
        # Cloud & infra
        "aws","gcp","azure","docker","kubernetes","terraform","ci/cd","github actions",
        "linux","unix","spark","hadoop","airflow","kafka","redis","elasticsearch",
        # Databases
        "postgresql","mysql","mongodb","sqlite","snowflake","bigquery","dbt",
        # Tools & practices
        "git","agile","scrum","rest api","graphql","microservices","oop",
        "unit testing","pytest","jira","confluence","figma",
        # Soft skills
        "communication","leadership","teamwork","problem solving","project management",
        "stakeholder management","analytical","research","collaboration",
    ]
    text_lower = raw_text.lower()
    found_skills = [kw.title() for kw in SKILL_KEYWORDS if kw in text_lower]

    # ── Roles: lines following common section headers ─────────────────────────
    roles = []
    role_section = False
    for line in lines:
        ll = line.lower()
        if any(h in ll for h in ["experience", "employment", "work history", "positions"]):
            role_section = True
            continue
        if role_section and any(h in ll for h in ["education", "skills", "projects", "certif"]):
            role_section = False
        if role_section and len(line) > 5 and len(line) < 80:
            roles.append({"title": line, "company": "", "years": 0})
        if len(roles) >= 5:
            break

    # ── Education ─────────────────────────────────────────────────────────────
    education = []
    for line in lines:
        if any(kw in line.lower() for kw in ["bachelor", "master", "phd", "b.s", "m.s",
                                               "b.e", "m.e", "degree", "university",
                                               "college", "institute", "diploma"]):
            education.append(line)

    # ── Summary ───────────────────────────────────────────────────────────────
    summary = " ".join(lines[1:4]) if len(lines) >= 4 else raw_text[:200]

    return {
        "name": name,
        "skills": found_skills,
        "roles": roles[:5],
        "education": education[:3],
        "summary": summary,
    }


# ── Tool 2: Job Requirements Fetcher ─────────────────────────────────────────

ONET_BASE = "https://services.onetcenter.org/ws/online/search"

def fetch_job_requirements(url: str = None, role: str = None) -> dict:
    """
    Tool 2 — Job Requirements Fetcher.

    Primary path:   fetch and parse a real job posting URL.
    Fallback path:  query O*NET for canonical skills for a role title.

    Returns:
        {
            "source": "url" | "onet",
            "role": str,
            "required_skills": [str, ...],
            "preferred_skills": [str, ...],
            "raw_description": str
        }
    """
    if url:
        return _fetch_from_url(url, role)
    elif role:
        return _fetch_from_onet(role)
    else:
        raise ValueError("fetch_job_requirements requires either a url or a role.")


def _fetch_from_url(url: str, role: str = None) -> dict:
    """Scrape a job posting URL and extract skills using the LLM."""
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        # If URL fails, fall back to O*NET
        if role:
            result = _fetch_from_onet(role)
            result["_url_error"] = str(e)
            return result
        raise

    soup = BeautifulSoup(resp.text, "html.parser")

    # Remove nav, footer, scripts to reduce noise
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)[:5000]  # trim to fit context window

    prompt = f"""
You are a job description analyst. Extract skills from the job posting below.

Return a JSON object with:
- "role": the job title (string)
- "required_skills": must-have skills explicitly stated (list of strings)
- "preferred_skills": nice-to-have or bonus skills (list of strings)
- "raw_description": first 300 characters of the posting for reference (string)

Job posting text:
\"\"\"
{text}
\"\"\"
"""
    raw = _llm(prompt, expect_json=True)
    try:
        result = _parse_json_response(raw)
        # Guard: LLM sometimes returns a list instead of a dict
        if isinstance(result, list):
            result = {"required_skills": result, "preferred_skills": [], "role": role or "Unknown"}
        if not isinstance(result, dict):
            raise ValueError("LLM returned non-dict")
        result["source"] = "url"
        if "role" not in result or not result["role"]:
            result["role"] = role or "Unknown"
        return result
    except (json.JSONDecodeError, ValueError):
        # Fallback: extract skills with simple heuristics from the job text
        skills = _heuristic_extract_skills(text)
        return {
            "source": "url",
            "role": role or "Unknown",
            "required_skills": skills,
            "preferred_skills": [],
            "raw_description": text[:300],
        }


def _fetch_from_onet(role: str) -> dict:
    """Query O*NET for canonical skill requirements for a role title."""
    try:
        # Search O*NET for the occupation code
        search_resp = requests.get(
            ONET_BASE,
            params={"keyword": role, "start": 1, "end": 5},
            headers={"Accept": "application/json"},
            timeout=10
        )
        search_resp.raise_for_status()
        data = search_resp.json()

        occupations = data.get("occupation", [])
        if not occupations:
            return _onet_fallback(role)

        # Use the top match
        top = occupations[0]
        code = top["code"]
        title = top["title"]

        # Fetch skills for this occupation
        skills_resp = requests.get(
            f"https://services.onetcenter.org/ws/online/occupations/{code}/skills",
            headers={"Accept": "application/json"},
            timeout=10
        )
        skills_resp.raise_for_status()
        skills_data = skills_resp.json()

        skills = [
            item["name"]
            for item in skills_data.get("element", [])
        ]

        return {
            "source": "onet",
            "role": title,
            "required_skills": skills[:15],
            "preferred_skills": [],
            "raw_description": f"O*NET data for: {title} ({code})"
        }

    except Exception:
        return _onet_fallback(role)


def _heuristic_extract_skills(text: str) -> list:
    """Extract skill keywords from raw job posting text."""
    # Reuse heuristic parse logic directly
    result = _heuristic_parse(text)
    return result.get("skills", [])


# Role → common required skills map (used when LLM and O*NET both fail)
ROLE_SKILL_MAP = {
    "data analyst":       ["SQL","Python","Excel","Data Visualization","Statistics","Tableau","Communication"],
    "ml engineer":        ["Python","Machine Learning","TensorFlow","PyTorch","SQL","Docker","Git","Statistics"],
    "software engineer":  ["Python","Java","Git","REST API","SQL","Agile","Unit Testing","OOP"],
    "software developer": ["Python","Java","Git","REST API","SQL","Agile","Unit Testing","OOP"],
    "product manager":    ["Agile","Stakeholder Management","Communication","Roadmap","Data Analysis","Jira"],
    "data scientist":     ["Python","Machine Learning","Statistics","SQL","Pandas","NumPy","Communication"],
    "frontend developer": ["JavaScript","React","HTML","CSS","Git","REST API","TypeScript"],
    "backend developer":  ["Python","Node.js","SQL","REST API","Docker","Git","PostgreSQL"],
}

def _onet_fallback(role: str) -> dict:
    """
    Last-resort fallback: use static role→skills map, then LLM.
    """
    role_lower = role.lower().strip()
    # Try static map first
    for key, skills in ROLE_SKILL_MAP.items():
        if key in role_lower or role_lower in key:
            return {
                "source": "static_map",
                "role": role,
                "required_skills": skills,
                "preferred_skills": [],
                "raw_description": f"Static skill map for: {role}",
            }

    # Try LLM as last resort
    prompt = f'''List 10 skills required for "{role}". Return JSON only.
Format: {{"role": "{role}", "required_skills": ["skill1","skill2"], "preferred_skills": []}}'''
    try:
        raw = _llm(prompt, expect_json=True, max_tokens=200)
        result = _parse_json_response(raw)
        if isinstance(result, dict) and isinstance(result.get("required_skills"), list):
            result["source"] = "llm_fallback"
            return result
    except Exception:
        pass

    return {
        "source": "llm_fallback",
        "role": role,
        "required_skills": [],
        "preferred_skills": [],
        "raw_description": ""
    }


# ── Tool 3: Gap Analyzer ──────────────────────────────────────────────────────

def analyze_gap(resume_data: dict, job_data: dict) -> dict:
    """
    Tool 3 — Gap Analyzer.

    Compares the candidate's skills against job requirements.
    Uses the LLM for semantic matching (not just string equality),
    so "Python programming" matches "Python" and "scikit-learn".

    Returns:
        {
            "matched_skills": [str, ...],
            "missing_skills": [str, ...],
            "overlap_score": float,   # 0.0 – 1.0
            "recommendation": str
        }
    """
    # Guard against malformed inputs
    if not isinstance(resume_data, dict):
        resume_data = {}
    if not isinstance(job_data, dict):
        job_data = {}

    candidate_skills = resume_data.get("skills", [])
    required_skills  = job_data.get("required_skills", [])
    preferred_skills = job_data.get("preferred_skills", [])
    target_role      = job_data.get("role", "the target role")

    # If no skills on either side, return meaningful error state
    if not candidate_skills and not required_skills:
        return {
            "matched_skills": [],
            "missing_skills": [],
            "preferred_gaps": [],
            "overlap_score": 0.0,
            "recommendation": "Could not extract skills from resume or job posting.",
            "_parse_error": True,
        }

    # If candidate has skills but LLM not needed — do heuristic match first
    if candidate_skills and required_skills:
        c_lower = [s.lower() for s in candidate_skills]
        matched  = [s for s in required_skills if any(s.lower() in c or c in s.lower() for c in c_lower)]
        missing  = [s for s in required_skills if s not in matched]
        score    = len(matched) / len(required_skills) if required_skills else 0.0
        heuristic_result = {
            "matched_skills": matched,
            "missing_skills": missing,
            "preferred_gaps": [s for s in preferred_skills if s not in matched],
            "overlap_score":  round(score, 2),
            "recommendation": f"You match {len(matched)} of {len(required_skills)} required skills for {target_role}.",
        }
    else:
        heuristic_result = None

    prompt = f"""
You are a career coach performing a skill gap analysis.

Candidate skills:
{json.dumps(candidate_skills, indent=2)}

Required skills for {target_role}:
{json.dumps(required_skills, indent=2)}

Preferred skills for {target_role}:
{json.dumps(preferred_skills, indent=2)}

Instructions:
- Use SEMANTIC matching, not exact string matching.
  e.g. "Python" matches "Python programming" and "scripting with Python".
- "matched_skills": required skills the candidate already has (list of strings)
- "missing_skills": required skills the candidate lacks (list of strings)
- "preferred_gaps": preferred skills the candidate lacks (list of strings)
- "overlap_score": float between 0.0 and 1.0 representing matched / total required
- "recommendation": 2-sentence career advice based on this gap

Return valid JSON only.
"""
    try:
        raw = _llm(prompt, expect_json=True, max_tokens=400)
        result = _parse_json_response(raw)
        if isinstance(result, dict) and isinstance(result.get("matched_skills"), list):
            if "overlap_score" in result:
                result["overlap_score"] = float(result["overlap_score"])
            return result
    except Exception:
        pass

    # LLM failed — return heuristic result if available
    if heuristic_result:
        return heuristic_result
    return {
        "matched_skills": [],
        "missing_skills": required_skills,
        "preferred_gaps": preferred_skills,
        "overlap_score": 0.0,
        "recommendation": "Gap analysis used fallback matching.",
    }


# ── Tool 4: Resume Rewriter ───────────────────────────────────────────────────

def rewrite_resume(resume_data: dict, gap_data: dict, job_data: dict) -> str:
    """
    Tool 4 — Resume Rewriter.

    Builds the rewritten resume structurally from parsed data first,
    then optionally enriches the summary with the LLM.
    This ensures real content appears even if the LLM is slow or fails.

    Returns:
        str — plain text rewritten resume
    """
    target_role     = job_data.get("role", "Target Role")
    matched_skills  = gap_data.get("matched_skills", [])
    all_skills      = resume_data.get("skills", [])
    roles           = resume_data.get("roles", [])
    education       = resume_data.get("education", [])
    name            = resume_data.get("name", "Candidate")
    required_skills = job_data.get("required_skills", [])

    # ── Build summary: try LLM, fall back to template ─────────────────────
    summary_prompt = (
        f'Write a 3-sentence professional summary for {name} targeting "{target_role}". '
        f'They have these relevant skills: {", ".join(matched_skills[:8])}. '
        f'Be specific and professional. No placeholders. Plain text only.'
    )
    try:
        summary = _llm(summary_prompt, expect_json=False, max_tokens=150).strip()
        # Reject if LLM returned a template or placeholder
        if any(bad in summary for bad in ["[Your", "placeholder", "lorem", "Skill 1", "Skill 2"]):
            raise ValueError("LLM returned template text")
    except Exception:
        skills_str = ", ".join(matched_skills[:5]) if matched_skills else ", ".join(all_skills[:5])
        summary = (
            f"{name} is a results-driven professional targeting the role of {target_role}. "
            f"With demonstrated experience in {skills_str}, they bring strong analytical "
            f"and collaborative skills to drive organisational success."
        )

    # ── Build skills section ───────────────────────────────────────────────
    # Prioritise matched skills, then add remaining resume skills
    priority_skills = matched_skills[:]
    for s in all_skills:
        if s not in priority_skills:
            priority_skills.append(s)

    # ── Build experience section ───────────────────────────────────────────
    experience_lines = []
    for role in roles[:5]:
        if isinstance(role, dict):
            title   = role.get("title", "")
            company = role.get("company", "")
            years   = role.get("years", "")
            if title:
                line = f"  {title}"
                if company:
                    line += f" | {company}"
                if years:
                    line += f" ({years} yrs)"
                experience_lines.append(line)
                # Add a generic aligned bullet for each role
                relevant = [s for s in required_skills if s.lower() in title.lower()]
                if matched_skills:
                    experience_lines.append(
                        f"  • Applied {matched_skills[0]} and {matched_skills[1] if len(matched_skills)>1 else 'cross-functional collaboration'} "
                        f"to achieve team objectives."
                    )
                experience_lines.append("")
        elif isinstance(role, str) and role.strip():
            experience_lines.append(f"  {role}")
            experience_lines.append("")

    if not experience_lines:
        experience_lines = ["  (Experience details not extracted from resume)"]

    # ── Assemble final document ────────────────────────────────────────────
    lines = [
        f"{name}",
        "=" * len(name),
        "",
        f"TARGET ROLE: {target_role}",
        "",
        "PROFESSIONAL SUMMARY",
        "-" * 20,
        summary,
        "",
        "CORE SKILLS",
        "-" * 20,
        "  " + "  |  ".join(priority_skills[:12]) if priority_skills else "  (See resume for full skill list)",
        "",
        "SKILLS ALIGNED TO ROLE",
        "-" * 20,
    ]
    if matched_skills:
        for s in matched_skills:
            lines.append(f"  ✓ {s}")
    else:
        lines.append("  (Run gap analysis to see aligned skills)")
    lines += [
        "",
        "EXPERIENCE",
        "-" * 20,
    ] + experience_lines + [
        "EDUCATION",
        "-" * 20,
    ]
    for edu in education:
        lines.append(f"  {edu}")
    if not education:
        lines.append("  (Education details not extracted from resume)")

    lines += [
        "",
        "---",
        f"Resume tailored for: {target_role}",
        f"Skills overlap with role: {gap_data.get('overlap_score', 0):.0%}",
    ]

    return "\n".join(lines)


# ── Tool 5: Word Document Generator ──────────────────────────────────────────

def generate_docx(missing_skills: list, preferred_gaps: list,
                  target_role: str, candidate_name: str = "Candidate",
                  matched_skills: list = None, all_candidate_skills: list = None,
                  overlap_score: float = 0.0, job_source: str = "") -> bytes:
    """
    Tool 5 — Word Document Generator.

    Produces a formatted .docx skill gap report including:
    - Candidate skill snapshot (what they already have)
    - Skills matched to the role
    - Required missing skills with actionable suggestions
    - Preferred missing skills
    - Next steps

    Returns:
        bytes — raw .docx file bytes
    """
    from docx.shared import Inches, Pt, RGBColor
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

    matched_skills      = matched_skills or []
    all_candidate_skills = all_candidate_skills or []

    doc = Document()

    # ── Page margins (narrower for better use of space) ───────────────────────
    for section in doc.sections:
        section.top_margin    = Inches(0.8)
        section.bottom_margin = Inches(0.8)
        section.left_margin   = Inches(1.0)
        section.right_margin  = Inches(1.0)

    # ── Base style ────────────────────────────────────────────────────────────
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)

    # ── Helper: add a styled heading ──────────────────────────────────────────
    def add_section_heading(text, color_rgb=(0x1F, 0x49, 0x7D)):
        h = doc.add_heading(text, level=1)
        for run in h.runs:
            run.font.color.rgb = RGBColor(*color_rgb)
            run.font.size = Pt(13)
        return h

    def add_kv(label, value):
        p = doc.add_paragraph()
        run_label = p.add_run(f"{label}: ")
        run_label.bold = True
        run_label.font.size = Pt(11)
        p.add_run(str(value)).font.size = Pt(11)

    # ── Cover block ───────────────────────────────────────────────────────────
    title = doc.add_heading(f"Skill Gap Report", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in title.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)
        run.font.size = Pt(20)

    sub = doc.add_paragraph(f"Career alignment analysis for {target_role}")
    sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in sub.runs:
        run.font.color.rgb = RGBColor(0x55, 0x55, 0x55)
        run.font.size = Pt(12)

    doc.add_paragraph("")
    add_kv("Candidate", candidate_name)
    add_kv("Target role", target_role)
    add_kv("Skill overlap score", f"{overlap_score:.0%} ({len(matched_skills)} of {len(matched_skills)+len(missing_skills)} required skills matched)")
    if job_source:
        add_kv("Requirements source", job_source)
    doc.add_paragraph("")

    # ── Section 1: Skills you already have ───────────────────────────────────
    add_section_heading("✅  Skills You Already Have", color_rgb=(0x0F, 0x6E, 0x56))
    doc.add_paragraph(
        "The following skills from your resume align with the requirements for this role."
    )
    if matched_skills:
        t = doc.add_table(rows=1, cols=3)
        t.style = "Table Grid"
        # Fill in 3 columns
        chunk = [matched_skills[i:i+3] for i in range(0, len(matched_skills), 3)]
        # Remove header row — just fill cells
        t._tbl.remove(t.rows[0]._tr)
        for group in chunk:
            row = t.add_row().cells
            for i, skill in enumerate(group):
                row[i].text = f"✓  {skill}"
                for para in row[i].paragraphs:
                    for run in para.runs:
                        run.font.color.rgb = RGBColor(0x0F, 0x6E, 0x56)
    else:
        doc.add_paragraph("No matched skills identified in this analysis run.")

    doc.add_paragraph("")

    # ── Section 2: Required skills to develop ────────────────────────────────
    add_section_heading("❌  Required Skills to Develop", color_rgb=(0x99, 0x3C, 0x1D))
    doc.add_paragraph(
        "These skills are explicitly required for the role and were not found in your resume. "
        "Prioritise these before applying."
    )
    doc.add_paragraph("")

    if missing_skills:
        table = doc.add_table(rows=1, cols=3)
        table.style = "Table Grid"
        hdr = table.rows[0].cells
        hdr[0].text = "Missing Skill"
        hdr[1].text = "Priority"
        hdr[2].text = "How to Develop"
        for cell in hdr:
            for para in cell.paragraphs:
                for run in para.runs:
                    run.bold = True
                    run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
                # shade header row dark
                tc = cell._tc
                tcPr = tc.get_or_add_tcPr()
                shd = OxmlElement("w:shd")
                shd.set(qn("w:fill"), "1F497D")
                shd.set(qn("w:color"), "auto")
                shd.set(qn("w:val"), "clear")
                tcPr.append(shd)

        for i, skill in enumerate(missing_skills):
            row_cells = table.add_row().cells
            row_cells[0].text = skill
            row_cells[1].text = "High" if i < 3 else "Medium"
            row_cells[2].text = _get_action(skill, target_role)
    else:
        doc.add_paragraph("No required skill gaps — strong alignment with this role.")

    doc.add_paragraph("")

    # ── Section 3: Preferred skills ───────────────────────────────────────────
    if preferred_gaps:
        add_section_heading("⚡  Preferred Skills to Develop", color_rgb=(0x85, 0x4F, 0x0B))
        doc.add_paragraph(
            "These are bonus qualifications. Developing them will strengthen your candidacy "
            "but are not screening requirements."
        )
        for skill in preferred_gaps:
            doc.add_paragraph(f"• {skill}", style="List Bullet")
        doc.add_paragraph("")

    # ── Section 4: All candidate skills (full snapshot) ──────────────────────
    if all_candidate_skills:
        add_section_heading("📋  Your Full Skill Snapshot", color_rgb=(0x18, 0x5F, 0xA5))
        doc.add_paragraph(
            "All skills detected in your resume. Use this list to verify the analysis is "
            "working from accurate data."
        )
        chunk3 = [all_candidate_skills[i:i+3] for i in range(0, len(all_candidate_skills), 3)]
        t2 = doc.add_table(rows=len(chunk3), cols=3)
        t2.style = "Table Grid"
        for r_idx, group in enumerate(chunk3):
            for c_idx, skill in enumerate(group):
                t2.rows[r_idx].cells[c_idx].text = skill
        doc.add_paragraph("")

    # ── Section 5: Next steps ─────────────────────────────────────────────────
    add_section_heading("🗺  Recommended Next Steps", color_rgb=(0x53, 0x4A, 0xB7))
    steps = [
        f"Address the top 3 required missing skills for {target_role} first — these are likely screening criteria.",
        "For each missing skill, find one hands-on project, course, or certification that produces evidence.",
        "Update your resume to explicitly name each skill once you can demonstrate it.",
        "Re-run Consensus with your updated resume to track your improved overlap score.",
        "Apply once your overlap score reaches 70%+ for this role.",
    ]
    for step in steps:
        doc.add_paragraph(step, style="List Number")

    # ── Save to bytes ─────────────────────────────────────────────────────────
    buffer = BytesIO()
    doc.save(buffer)
    buffer.seek(0)
    return buffer.read()


def _get_action(skill: str, role: str) -> str:
    """Generate a short actionable suggestion for acquiring a skill."""
    prompt = f"""
In one short sentence (max 15 words), suggest how someone targeting a "{role}"
role could quickly demonstrate the skill: "{skill}".
Be specific and practical. No preamble.
"""
    return _llm(prompt, expect_json=False, max_tokens=80).strip().strip('"')


Overwriting /content/consensus/tools.py


In [17]:
%%writefile /content/consensus/agent.py

"""
agent.py — Consensus Career Strategy Agent
Hand-written agent loop. No frameworks (no LangChain, CrewAI, etc.)

Loop design:
    The agent maintains a state dict and a scratchpad of observations.
    Each iteration, the planner (LLM) reads the scratchpad and decides
    which tool to call next. The loop continues until the planner signals
    DONE or max_steps is reached.
"""

import json
import requests
from tools import (
    parse_resume,
    fetch_job_requirements,
    analyze_gap,
    rewrite_resume,
    generate_docx,
)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL      = "llama3.2:3b"

# ── Available tools registry ──────────────────────────────────────────────────
# The planner sees these descriptions and picks the next one to call.

TOOLS = {
    "parse_resume": {
        "description": "Parse the uploaded resume PDF and extract structured skills, roles, and education.",
        "requires": ["pdf_bytes"],
    },
    "fetch_job_requirements": {
        "description": "Fetch required skills from a job posting URL or O*NET if no URL is given.",
        "requires": ["url_or_none", "role"],
    },
    "analyze_gap": {
        "description": "Compare candidate skills against job requirements and identify gaps.",
        "requires": ["resume_data", "job_data"],
    },
    "rewrite_resume": {
        "description": "Rewrite the resume to align with the target role using job-relevant language.",
        "requires": ["resume_data", "gap_data", "job_data"],
    },
    "generate_docx": {
        "description": "Generate a Word document listing missing skills and suggested actions.",
        "requires": ["missing_skills", "preferred_gaps", "target_role", "candidate_name"],
    },
}

TOOL_ORDER = [
    "parse_resume",
    "fetch_job_requirements",
    "analyze_gap",
    "rewrite_resume",
    "generate_docx",
]


# ── Planner ───────────────────────────────────────────────────────────────────

def plan_next_step(scratchpad: list[dict], available_tools: list[str]) -> str:
    """
    Ask the LLM which tool to call next given the current scratchpad.

    Returns the tool name string, or "DONE" if the agent should stop.
    """
    scratchpad_text = "\n".join(
        f"Step {i+1} — {s['tool']}: {s['summary']}"
        for i, s in enumerate(scratchpad)
    )

    tools_text = "\n".join(
        f"- {name}: {TOOLS[name]['description']}"
        for name in available_tools
    )

    prompt = f"""
You are the planning module of a career strategy agent.

Completed steps so far:
{scratchpad_text if scratchpad_text else "None yet."}

Available tools (only these names are valid):
{tools_text}

Which single tool should be called next to make progress toward the final goal
of: (1) a rewritten resume and (2) a Word document of missing skills?

Reply with ONLY the tool name from the list above, or reply DONE if all outputs
are complete. No explanation.
""".strip()

    payload = {
        "model":  MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 32},
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=60)
    resp.raise_for_status()
    answer = resp.json()["response"].strip().split()[0]  # take first word only

    # Validate — if the LLM hallucinates a tool name, fall back to order
    if answer in available_tools or answer == "DONE":
        return answer
    return available_tools[0] if available_tools else "DONE"


# ── Agent loop ────────────────────────────────────────────────────────────────

def run_agent(
    pdf_bytes: bytes,
    target_role: str,
    job_url: str = None,
    max_steps: int = 10,
    progress_callback=None,   # callable(step_name, message) for UI updates
) -> dict:
    """
    Main agent loop.

    Args:
        pdf_bytes:          Raw bytes of the uploaded resume PDF.
        target_role:        The role the user is targeting (e.g. "Data Analyst").
        job_url:            Optional URL to a specific job posting.
        max_steps:          Safety cap on loop iterations.
        progress_callback:  Optional function(tool_name, message) for live UI updates.

    Returns:
        {
            "resume_data":     dict,   # parsed resume
            "job_data":        dict,   # job requirements
            "gap_data":        dict,   # gap analysis
            "rewritten_resume": str,   # plain text rewritten resume
            "docx_bytes":      bytes,  # Word document
            "scratchpad":      list,   # full agent trace
            "steps_taken":     int,
        }
    """

    def _log(tool: str, msg: str):
        if progress_callback:
            progress_callback(tool, msg)

    # ── Shared state ──────────────────────────────────────────────────────────
    state = {
        "pdf_bytes":        pdf_bytes,
        "target_role":      target_role,
        "job_url":          job_url,
        "resume_data":      None,
        "job_data":         None,
        "gap_data":         None,
        "rewritten_resume": None,
        "docx_bytes":       None,
    }

    scratchpad: list[dict] = []
    remaining_tools = list(TOOL_ORDER)  # tools not yet called
    steps = 0

    # ── Loop ──────────────────────────────────────────────────────────────────
    while steps < max_steps and remaining_tools:
        steps += 1

        # 1. PLAN — always take the next tool in order.
        # We use the LLM planner as a confirmation step only; if it returns
        # something unexpected we fall back to the next tool in sequence.
        # This prevents the planner from jumping ahead before state is ready.
        next_tool = remaining_tools[0]

        _log(next_tool, f"Calling tool: {next_tool}…")

        # 2. EXECUTE — call the chosen tool
        try:
            observation, summary = _execute_tool(next_tool, state)
        except Exception as e:
            summary = f"ERROR in {next_tool}: {e}"
            observation = None
            _log(next_tool, summary)

        # 3. OBSERVE — store result in state and scratchpad
        _store_observation(next_tool, observation, state)
        scratchpad.append({"tool": next_tool, "summary": summary})
        remaining_tools.remove(next_tool)

        _log(next_tool, f"✓ {summary}")

    return {
        "resume_data":      state["resume_data"],
        "job_data":         state["job_data"],
        "gap_data":         state["gap_data"],
        "rewritten_resume": state["rewritten_resume"],
        "docx_bytes":       state["docx_bytes"],
        "scratchpad":       scratchpad,
        "steps_taken":      steps,
    }


# ── Tool execution dispatcher ─────────────────────────────────────────────────

def _execute_tool(tool_name: str, state: dict) -> tuple:
    """
    Call the appropriate tool function and return (result, summary_string).
    """
    if tool_name == "parse_resume":
        result   = parse_resume(state["pdf_bytes"])
        name     = result.get("name", "Candidate")
        n_skills = len(result.get("skills", []))
        summary  = f"Parsed resume for {name}. Found {n_skills} skills."
        return result, summary

    elif tool_name == "fetch_job_requirements":
        result   = fetch_job_requirements(
            url=state.get("job_url"),
            role=state["target_role"]
        )
        source   = result.get("source", "unknown")
        role     = result.get("role", state["target_role"])
        n_skills = len(result.get("required_skills", []))
        summary  = (
            f"Fetched {n_skills} required skills for '{role}' "
            f"(source: {source})."
        )
        return result, summary

    elif tool_name == "analyze_gap":
        if state["resume_data"] is None:
            raise RuntimeError("resume_data is not ready — parse_resume must run first.")
        if state["job_data"] is None:
            raise RuntimeError("job_data is not ready — fetch_job_requirements must run first.")
        result  = analyze_gap(state["resume_data"], state["job_data"])
        matched = len(result.get("matched_skills", []))
        missing = len(result.get("missing_skills", []))
        score   = result.get("overlap_score", 0.0)
        summary = (
            f"Gap analysis: {matched} matched, {missing} missing. "
            f"Overlap score: {score:.0%}."
        )
        return result, summary

    elif tool_name == "rewrite_resume":
        if state["resume_data"] is None:
            raise RuntimeError("resume_data is not ready — parse_resume must run first.")
        if state["job_data"] is None:
            raise RuntimeError("job_data is not ready — fetch_job_requirements must run first.")
        if state["gap_data"] is None:
            raise RuntimeError("gap_data is not ready — analyze_gap must run first.")
        result  = rewrite_resume(
            state["resume_data"],
            state["gap_data"],
            state["job_data"]
        )
        summary = f"Resume rewritten ({len(result.split())} words)."
        return result, summary

    elif tool_name == "generate_docx":
        if state["gap_data"] is None:
            raise RuntimeError("gap_data is not ready — analyze_gap must run first.")
        if state["job_data"] is None:
            raise RuntimeError("job_data is not ready — fetch_job_requirements must run first.")
        if state["resume_data"] is None:
            raise RuntimeError("resume_data is not ready — parse_resume must run first.")
        gap_data     = state["gap_data"]
        resume_data  = state["resume_data"]
        job_data     = state["job_data"]
        result       = generate_docx(
            missing_skills       = gap_data.get("missing_skills", []),
            preferred_gaps       = gap_data.get("preferred_gaps", []),
            target_role          = job_data.get("role", state["target_role"]),
            candidate_name       = resume_data.get("name", "Candidate"),
            matched_skills       = gap_data.get("matched_skills", []),
            all_candidate_skills = resume_data.get("skills", []),
            overlap_score        = gap_data.get("overlap_score", 0.0),
            job_source           = job_data.get("source", ""),
        )
        summary = f"Word document generated ({len(result):,} bytes)."
        return result, summary

    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def _store_observation(tool_name: str, observation, state: dict):
    """Write tool output back into the shared state dict."""
    mapping = {
        "parse_resume":        "resume_data",
        "fetch_job_requirements": "job_data",
        "analyze_gap":         "gap_data",
        "rewrite_resume":      "rewritten_resume",
        "generate_docx":       "docx_bytes",
    }
    key = mapping.get(tool_name)
    if key and observation is not None:
        state[key] = observation

Overwriting /content/consensus/agent.py


In [18]:
%%writefile /content/consensus/app.py
"""
app.py — Consensus Career Strategy Agent
Streamlit user interface.

Run with:
    streamlit run app.py
"""

import streamlit as st
import json
from agent import run_agent

# ── Page config ───────────────────────────────────────────────────────────────

st.set_page_config(
    page_title="Consensus — Career Strategy Agent",
    page_icon="🎯",
    layout="wide",
)

# ── Custom CSS ────────────────────────────────────────────────────────────────

st.markdown("""
<style>
    .main { background-color: #f8f9fa; }
    .stProgress > div > div { background-color: #1F497D; }
    .metric-card {
        background: white;
        border-radius: 10px;
        padding: 1rem 1.5rem;
        border-left: 4px solid #1F497D;
        margin-bottom: 0.5rem;
    }
    .skill-pill {
        display: inline-block;
        background: #e8f0fe;
        color: #1F497D;
        border-radius: 20px;
        padding: 0.25rem 0.75rem;
        margin: 0.2rem;
        font-size: 0.85rem;
    }
    .missing-pill {
        display: inline-block;
        background: #fce8e6;
        color: #c5221f;
        border-radius: 20px;
        padding: 0.25rem 0.75rem;
        margin: 0.2rem;
        font-size: 0.85rem;
    }
    .step-log {
        background: #1e1e1e;
        color: #d4d4d4;
        border-radius: 8px;
        padding: 1rem;
        font-family: monospace;
        font-size: 0.85rem;
        max-height: 200px;
        overflow-y: auto;
    }
</style>
""", unsafe_allow_html=True)


# ── Header ────────────────────────────────────────────────────────────────────

st.title("🎯 Consensus")
st.markdown("**Career Strategy Agent** — upload your resume, set your target, get your plan.")
st.divider()


# ── Sidebar: inputs ───────────────────────────────────────────────────────────

with st.sidebar:
    st.header("Your Details")

    uploaded_file = st.file_uploader(
        "Upload Resume (PDF)",
        type=["pdf"],
        help="Your current resume as a PDF file."
    )

    target_role = st.text_input(
        "Target Role",
        placeholder="e.g. Data Analyst, Product Manager",
        help="The job title you're aiming for."
    )

    job_url = st.text_input(
        "Job Posting URL (optional)",
        placeholder="https://www.linkedin.com/jobs/view/...",
        help="Paste a specific job posting for tailored analysis. "
             "Leave blank to use O*NET data for your role."
    )

    run_button = st.button(
        "▶  Run Consensus",
        type="primary",
        disabled=not (uploaded_file and target_role),
        use_container_width=True
    )

    st.divider()
    st.caption("Built with Ollama · llama3.2:3b · O*NET · python-docx")


# ── Main area: results ────────────────────────────────────────────────────────

if not run_button:
    # Landing state
    col1, col2, col3 = st.columns(3)
    with col1:
        st.info("**Step 1**\nUpload your resume PDF in the sidebar.")
    with col2:
        st.info("**Step 2**\nEnter your target role and optionally paste a job URL.")
    with col3:
        st.info("**Step 3**\nClick Run — the agent will analyse, compare, and rewrite.")
    st.stop()


# ── Run the agent ─────────────────────────────────────────────────────────────

pdf_bytes = uploaded_file.read()

# Live progress display
st.subheader("Agent Running…")
progress_bar  = st.progress(0)
status_text   = st.empty()
log_container = st.empty()
agent_log     = []

TOOL_STEPS = {
    "parse_resume":           (1, "Parsing resume…"),
    "fetch_job_requirements": (2, "Fetching job requirements…"),
    "analyze_gap":            (3, "Analysing skill gap…"),
    "rewrite_resume":         (4, "Rewriting resume…"),
    "generate_docx":          (5, "Generating Word document…"),
    "DONE":                   (5, "Complete!"),
}
TOTAL_STEPS = 5

def on_progress(tool_name: str, message: str):
    step_num, label = TOOL_STEPS.get(tool_name, (0, tool_name))
    progress_bar.progress(step_num / TOTAL_STEPS)
    status_text.markdown(f"**{label}** — {message}")
    agent_log.append(f"[{tool_name}] {message}")
    log_container.markdown(
        "<div class='step-log'>" +
        "<br>".join(agent_log[-12:]) +
        "</div>",
        unsafe_allow_html=True
    )

with st.spinner(""):
    results = run_agent(
        pdf_bytes         = pdf_bytes,
        target_role       = target_role,
        job_url           = job_url.strip() if job_url.strip() else None,
        progress_callback = on_progress,
    )

progress_bar.progress(1.0)
status_text.success("✅ Analysis complete!")
log_container.empty()

st.divider()


# ── Results display ───────────────────────────────────────────────────────────

resume_data = results.get("resume_data") or {}
job_data    = results.get("job_data")    or {}
gap_data    = results.get("gap_data")    or {}

tab1, tab2, tab3, tab4 = st.tabs([
    "📊 Skill Gap",
    "📄 Rewritten Resume",
    "📥 Downloads",
    "🔍 Agent Trace",
])

# ── Tab 1: Skill Gap ──────────────────────────────────────────────────────────
with tab1:
    col_left, col_right = st.columns([1, 2])

    with col_left:
        score = gap_data.get("overlap_score", 0.0)
        st.metric(
            label="Skill Overlap Score",
            value=f"{score:.0%}",
            delta=f"{len(gap_data.get('matched_skills', []))} matched of "
                  f"{len(job_data.get('required_skills', []))} required",
        )
        st.markdown(f"**Role:** {job_data.get('role', target_role)}")
        st.markdown(f"**Source:** {job_data.get('source', '—')}")

        st.markdown("**Recommendation:**")
        st.info(gap_data.get("recommendation", "—"))

    with col_right:
        matched  = gap_data.get("matched_skills", [])
        missing  = gap_data.get("missing_skills", [])
        preferred = gap_data.get("preferred_gaps", [])

        st.markdown("**✅ Skills you already have:**")
        if matched:
            pills = " ".join(
                f"<span class='skill-pill'>{s}</span>" for s in matched
            )
            st.markdown(pills, unsafe_allow_html=True)
        else:
            st.caption("None matched.")

        st.markdown("")
        st.markdown("**❌ Required skills to develop:**")
        if missing:
            pills = " ".join(
                f"<span class='missing-pill'>{s}</span>" for s in missing
            )
            st.markdown(pills, unsafe_allow_html=True)
        elif gap_data.get("_parse_error"):
            st.warning("Gap analysis could not complete — check the Agent Trace tab for details.")
        elif score > 0:
            st.success("No required gaps — strong match!")
        else:
            st.warning("No skills were matched. Check that your resume PDF is text-readable (not a scanned image).")

        if preferred:
            st.markdown("")
            st.markdown("**⚡ Preferred skills to develop:**")
            pills = " ".join(
                f"<span class='skill-pill'>{s}</span>" for s in preferred
            )
            st.markdown(pills, unsafe_allow_html=True)


# ── Tab 2: Rewritten Resume ───────────────────────────────────────────────────
with tab2:
    rewritten = results.get("rewritten_resume", "")
    if rewritten:
        st.markdown("Your resume has been rewritten to align with "
                    f"**{job_data.get('role', target_role)}**.")
        st.text_area(
            label="Rewritten Resume",
            value=rewritten,
            height=500,
            label_visibility="collapsed"
        )
    else:
        st.warning("Resume rewrite not available.")


# ── Tab 3: Downloads ──────────────────────────────────────────────────────────
with tab3:
    st.markdown("### Download your outputs")
    col_a, col_b = st.columns(2)

    with col_a:
        docx_bytes = results.get("docx_bytes")
        if docx_bytes:
            st.download_button(
                label     = "📥 Download Skill Gap Report (.docx)",
                data      = docx_bytes,
                file_name = f"skill_gap_{target_role.replace(' ', '_')}.docx",
                mime      = "application/vnd.openxmlformats-officedocument"
                            ".wordprocessingml.document",
                use_container_width=True,
            )
        else:
            st.warning("Word document not generated.")

    with col_b:
        if rewritten:
            st.download_button(
                label     = "📥 Download Rewritten Resume (.txt)",
                data      = rewritten,
                file_name = f"resume_{target_role.replace(' ', '_')}.txt",
                mime      = "text/plain",
                use_container_width=True,
            )


# ── Tab 4: Agent Trace ────────────────────────────────────────────────────────
with tab4:
    st.markdown("### Agent scratchpad")
    st.caption("Each step shows what the agent did and what it observed.")
    scratchpad = results.get("scratchpad", [])
    for i, step in enumerate(scratchpad):
        with st.expander(f"Step {i+1} — {step['tool']}", expanded=(i == 0)):
            st.markdown(step["summary"])

    st.markdown(f"**Total steps taken:** {results.get('steps_taken', 0)}")

    with st.expander("Full state (JSON)"):
        safe_results = {
            k: v for k, v in results.items()
            if k not in ("docx_bytes", "pdf_bytes")
        }
        st.json(safe_results)


Overwriting /content/consensus/app.py


In [25]:
%%writefile /content/consensus/eval.py

"""
eval.py — Consensus Agent Evaluation
Quantitative performance measurement across all tools.

Test set:
  resume_01.pdf — Priya Nair, HR Manager (synthetic, known ground truth)
  resume_02.pdf — Sharmil Kallichanda, AI/ML / Cloud Consultant
  resume_03.pdf — Somaiah Poonacha, Planning & Forecasting Manager

Run with:
    python eval.py

Outputs:
  - Formatted table printed to console
  - eval_results.json saved to current directory
"""

import json
import time
from pathlib import Path

# ── Ground truth ──────────────────────────────────────────────────────────────

GROUND_TRUTH = [
    {
        "id": "test_01",
        "name": "Priya Nair — HR Manager",
        "resume_path": "eval_data/resume_01.pdf",
        "true_skills": [
            "Talent Acquisition", "Recruitment", "Employee Relations",
            "Performance Management", "HR Policy Development",
            "Onboarding", "Compensation", "Benefits Administration",
            "Workday", "BambooHR", "Learning and Development",
            "Diversity Equity Inclusion", "Employment Law", "Compliance",
            "Stakeholder Management", "SHRM-CP", "PHR",
        ],
        "target_role": "HR Manager",
        "job_url": None,
        "true_required_skills": [
            "Talent Acquisition", "Employee Relations", "Performance Management",
            "HR Policy", "Compensation", "Benefits", "HRIS",
            "Compliance", "Stakeholder Management", "Communication",
        ],
        "true_matched": [
            "Talent Acquisition", "Employee Relations", "Performance Management",
            "Compensation", "Benefits", "HRIS", "Compliance", "Stakeholder Management",
        ],
        "true_missing": ["HR Policy", "Communication"],
        "target_keywords": [
            "talent", "recruitment", "employee relations", "performance",
            "hr", "compliance", "hris", "workday", "onboarding", "benefits",
        ],
    },
    {
        "id": "test_02",
        "name": "Sharmil Kallichanda — ML Engineer",
        "resume_path": "eval_data/resume_02.pdf",
        "true_skills": [
            "Python", "SQL", "Machine Learning", "Deep Learning",
            "LLM", "NLP", "Computer Vision", "RAG",
            "PyTorch", "TensorFlow", "scikit-learn", "XGBoost",
            "LangGraph", "MLflow", "LangFuse", "Prompt Engineering",
            "SHAP", "CI/CD", "Oracle Cloud", "Data Migration",
            "Agile", "ERP", "OpenAI APIs", "ChromaDB",
        ],
        "target_role": "ML Engineer",
        "job_url": None,
        "true_required_skills": [
            "Python", "Machine Learning", "Deep Learning", "PyTorch",
            "TensorFlow", "SQL", "MLOps", "CI/CD", "Git",
            "Statistics", "Data Engineering",
        ],
        "true_matched": [
            "Python", "Machine Learning", "Deep Learning",
            "PyTorch", "TensorFlow", "SQL", "CI/CD",
        ],
        "true_missing": ["MLOps", "Git", "Statistics", "Data Engineering"],
        "target_keywords": [
            "machine learning", "python", "pytorch", "tensorflow",
            "nlp", "llm", "deep learning", "mlops", "model", "ai",
        ],
    },
    {
        "id": "test_03",
        "name": "Somaiah Poonacha — Supply Chain / Data Analyst",
        "resume_path": "eval_data/resume_03.pdf",
        "true_skills": [
            "Supply Chain", "Demand Planning", "Forecasting",
            "S&OP", "Inventory Management", "Vendor Management",
            "SAP MM", "NetSuite", "Sage 100", "Tableau",
            "Microsoft SQL", "MS Project", "JIRA", "ERP Implementation",
            "Data Analysis", "Reporting", "Stakeholder Management",
            "Communication", "Leadership",
        ],
        "target_role": "Data Analyst",
        "job_url": None,
        "true_required_skills": [
            "SQL", "Python", "Data Visualization", "Excel",
            "Statistical Analysis", "Communication",
            "Problem Solving", "Tableau", "Reporting",
        ],
        "true_matched": [
            "SQL", "Data Visualization", "Communication", "Tableau", "Reporting",
        ],
        "true_missing": [
            "Python", "Excel", "Statistical Analysis", "Problem Solving",
        ],
        "target_keywords": [
            "data", "analyst", "sql", "tableau", "reporting",
            "analysis", "insight", "visualization", "forecast", "supply chain",
        ],
    },
]


# ── Metric helpers ────────────────────────────────────────────────────────────

def precision_recall_f1(predicted, ground_truth):
    if not predicted and not ground_truth:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "tp": 0, "fp": 0, "fn": 0}
    if not predicted:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "fp": 0, "fn": len(ground_truth)}
    if not ground_truth:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "fp": len(predicted), "fn": 0}

    pred_lower = [p.lower() for p in predicted]
    true_lower = [t.lower() for t in ground_truth]

    def is_match(a, b_list):
        return any(a in b or b in a for b in b_list)

    tp = sum(1 for p in pred_lower if is_match(p, true_lower))
    fp = sum(1 for p in pred_lower if not is_match(p, true_lower))
    fn = sum(1 for t in true_lower if not is_match(t, pred_lower))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    return {
        "precision": round(precision, 3),
        "recall":    round(recall, 3),
        "f1":        round(f1, 3),
        "tp": tp, "fp": fp, "fn": fn,
    }


def keyword_match_rate(text, keywords):
    if not keywords:
        return 0.0
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return round(hits / len(keywords), 3)


def load_pdf(path):
    p = Path(path)
    if not p.exists():
        return None
    with open(p, "rb") as f:
        return f.read()


# ── Per-tool evaluators ───────────────────────────────────────────────────────

def eval_tool1(test, parse_resume_fn):
    pdf = load_pdf(test["resume_path"])
    if pdf is None:
        return {"skipped": True, "reason": f"PDF not found: {test['resume_path']}"}
    start = time.time()
    result = parse_resume_fn(pdf)
    latency = round(time.time() - start, 2)
    predicted = result.get("skills", [])
    metrics = precision_recall_f1(predicted, test["true_skills"])
    metrics.update({
        "latency_s":       latency,
        "n_predicted":     len(predicted),
        "n_ground_truth":  len(test["true_skills"]),
        "name_extracted":  result.get("name", ""),
        "predicted_skills": predicted,
    })
    return metrics, result


def eval_tool2(test, fetch_fn):
    start = time.time()
    result = fetch_fn(url=test.get("job_url"), role=test["target_role"])
    latency = round(time.time() - start, 2)
    fetched = result.get("required_skills", [])
    metrics = precision_recall_f1(fetched, test["true_required_skills"])
    metrics.update({
        "latency_s":    latency,
        "source":       result.get("source", "unknown"),
        "n_fetched":    len(fetched),
        "fetched_skills": fetched,
    })
    return metrics, result


def eval_tool3(test, gap_fn, resume_data, job_data):
    start = time.time()
    result = gap_fn(resume_data, job_data)
    latency = round(time.time() - start, 2)
    missing_metrics = precision_recall_f1(result.get("missing_skills", []), test["true_missing"])
    matched_metrics = precision_recall_f1(result.get("matched_skills", []), test["true_matched"])
    return {
        "missing_precision": missing_metrics["precision"],
        "missing_recall":    missing_metrics["recall"],
        "missing_f1":        missing_metrics["f1"],
        "matched_precision": matched_metrics["precision"],
        "matched_recall":    matched_metrics["recall"],
        "matched_f1":        matched_metrics["f1"],
        "overlap_score":     result.get("overlap_score", 0.0),
        "latency_s":         latency,
        "predicted_missing": result.get("missing_skills", []),
        "predicted_matched": result.get("matched_skills", []),
    }, result


def eval_tool4(test, rewrite_fn, resume_data, gap_data, job_data):
    start = time.time()
    rewritten = rewrite_fn(resume_data, gap_data, job_data)
    latency = round(time.time() - start, 2)
    kw_rate = keyword_match_rate(rewritten, test["target_keywords"])
    has_placeholder = any(ph in rewritten for ph in ["[Your", "Skill 1", "Skill 2", "lorem"])
    return {
        "keyword_match_rate": kw_rate,
        "word_count":         len(rewritten.split()),
        "latency_s":          round(latency, 2),
        "has_placeholder":    has_placeholder,
        "rewritten_preview":  rewritten[:300].replace("\n", " "),
    }


# ── Console table printer ─────────────────────────────────────────────────────

def print_results_table(all_results):
    SEP   = "=" * 72
    INNER = "-" * 72

    print(f"\n{SEP}")
    print("  CONSENSUS AGENT — QUANTITATIVE EVALUATION REPORT")
    print(SEP)

    for res in all_results:
        print(f"\n  [{res['id']}]  {res['name']}  →  Target: {res['role']}")
        print(INNER)

        def row(label, value, note=""):
            note_str = f"  ({note})" if note else ""
            print(f"  {label:<40} {value}{note_str}")

        t1 = res.get("tool1_parse", {})
        if t1.get("skipped"):
            row("Tool 1 — Resume parser", "SKIPPED", t1.get("reason", ""))
        else:
            row("Tool 1 — Resume parser",
                f"F1={t1.get('f1',0):.2f}",
                f"P={t1.get('precision',0):.2f}  R={t1.get('recall',0):.2f}  "
                f"{t1.get('n_predicted',0)} skills  {t1.get('latency_s',0)}s")

        t2 = res.get("tool2_fetch", {})
        row("Tool 2 — Job requirements fetch",
            f"F1={t2.get('f1',0):.2f}",
            f"P={t2.get('precision',0):.2f}  R={t2.get('recall',0):.2f}  "
            f"src={t2.get('source','')}  {t2.get('latency_s',0)}s")

        t3 = res.get("tool3_gap", {})
        row("Tool 3 — Gap (missing skills)",
            f"F1={t3.get('missing_f1',0):.2f}",
            f"P={t3.get('missing_precision',0):.2f}  R={t3.get('missing_recall',0):.2f}")
        row("Tool 3 — Gap (matched skills)",
            f"F1={t3.get('matched_f1',0):.2f}",
            f"overlap={t3.get('overlap_score',0):.0%}  {t3.get('latency_s',0)}s")

        t4 = res.get("tool4_rewrite", {})
        warn = " ⚠ placeholder" if t4.get("has_placeholder") else ""
        row("Tool 4 — Resume rewriter (ATS)",
            f"{t4.get('keyword_match_rate',0):.0%}",
            f"{t4.get('word_count',0)} words  {t4.get('latency_s',0)}s{warn}")

    # Aggregates
    def avg(key, subkey):
        vals = [r[key][subkey] for r in all_results
                if key in r and not r[key].get("skipped") and subkey in r[key]]
        return sum(vals) / len(vals) if vals else None

    print(f"\n{SEP}")
    print("  AGGREGATE AVERAGES")
    print(SEP)
    print(f"\n  {'Metric':<40} {'Avg Score':>10}  Label")
    print(f"  {'-'*40} {'-'*10}  {'-'*10}")

    rows = [
        ("Tool 1 — Resume parser",    "tool1_parse",   "f1",                "F1"),
        ("Tool 2 — Job fetch",        "tool2_fetch",   "f1",                "F1"),
        ("Tool 3 — Gap (missing)",    "tool3_gap",     "missing_f1",        "F1"),
        ("Tool 3 — Gap (matched)",    "tool3_gap",     "matched_f1",        "F1"),
        ("Tool 4 — Resume rewriter",  "tool4_rewrite", "keyword_match_rate","ATS Match"),
    ]
    for label, tk, mk, ml in rows:
        v = avg(tk, mk)
        print(f"  {label:<40} {f'{v:.2%}' if v is not None else 'N/A':>10}  {ml}")

    print(f"\n{SEP}\n")


# ── Main ──────────────────────────────────────────────────────────────────────

def run_evaluation():
    from tools import parse_resume, fetch_job_requirements, analyze_gap, rewrite_resume

    all_results = []

    for test in GROUND_TRUTH:
        print(f"\n  Running {test['id']} — {test['name']}")
        res = {"id": test["id"], "name": test["name"], "role": test["target_role"]}

        print("    [1/4] Parsing resume...")
        t1_metrics, resume_data = eval_tool1(test, parse_resume)
        res["tool1_parse"] = t1_metrics
        if t1_metrics.get("skipped"):
            print(f"    Skipped: {t1_metrics['reason']}")
            all_results.append(res)
            continue

        print("    [2/4] Fetching job requirements...")
        t2_metrics, job_data = eval_tool2(test, fetch_job_requirements)
        res["tool2_fetch"] = t2_metrics

        print("    [3/4] Analysing skill gap...")
        t3_metrics, gap_data = eval_tool3(test, analyze_gap, resume_data, job_data)
        res["tool3_gap"] = t3_metrics

        print("    [4/4] Rewriting resume...")
        t4_metrics = eval_tool4(test, rewrite_resume, resume_data, gap_data, job_data)
        res["tool4_rewrite"] = t4_metrics

        all_results.append(res)
        print("    Done.")

    # Print table to console
    print_results_table(all_results)

    # Save to JSON
    def make_serialisable(obj):
        if isinstance(obj, dict):
            return {k: make_serialisable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [make_serialisable(i) for i in obj]
        if isinstance(obj, (int, float, str, bool)) or obj is None:
            return obj
        return str(obj)

    out_path = "eval_results.json"
    with open(out_path, "w") as f:
        json.dump(make_serialisable(all_results), f, indent=2)

    print(f"  Saved to: {out_path}")
    return all_results


if __name__ == "__main__":
    run_evaluation()

Overwriting /content/consensus/eval.py


In [12]:
# ── 7. Launch Streamlit via ngrok ────────────────────────────────────────────
# Replace YOUR_NGROK_TOKEN with your free token from https://ngrok.com
from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = '3C0NNuZs3WaCc1CulBVY8xQeWrn_673S5BnzwF6vUTsX5pShn'  # <-- paste your token here
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in background
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', '/content/consensus/app.py',
     '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f'\n✅ Consensus is live at: {public_url}')
print('Open that URL in your browser.')


✅ Consensus is live at: NgrokTunnel: "https://odis-polyprotic-bethel.ngrok-free.dev" -> "http://localhost:8501"
Open that URL in your browser.


Consensus is live at: NgrokTunnel: "https://odis-polyprotic-bethel.ngrok-free.dev" -> "http://localhost:8501"
Open that URL in your browser.

Load and run Evaluation

In [23]:
from google.colab import files
uploaded = files.upload()  # a file picker dialog will appear

Saving eval_data.zip to eval_data (1).zip


In [24]:
!unzip eval_data.zip -d /content/consensus

Archive:  eval_data.zip
replace /content/consensus/eval_data/resume_01.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/consensus/eval_data/resume_01.pdf  
replace /content/consensus/eval_data/resume_02.pdf.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/consensus/eval_data/resume_02.pdf.pdf  
replace /content/consensus/eval_data/resume_03.pdf.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/consensus/eval_data/resume_03.pdf.pdf  


In [27]:
# ── Run evaluation + print results + save JSON ────────────────────────────────
%cd /content/consensus

from eval import run_evaluation

results = run_evaluation()

/content/consensus

  Running test_01 — Priya Nair — HR Manager
    [1/4] Parsing resume...
    [2/4] Fetching job requirements...
    [3/4] Analysing skill gap...
    [4/4] Rewriting resume...
    Done.

  Running test_02 — Sharmil Kallichanda — ML Engineer
    [1/4] Parsing resume...
    [2/4] Fetching job requirements...
    [3/4] Analysing skill gap...
    [4/4] Rewriting resume...
    Done.

  Running test_03 — Somaiah Poonacha — Supply Chain / Data Analyst
    [1/4] Parsing resume...
    [2/4] Fetching job requirements...
    [3/4] Analysing skill gap...
    [4/4] Rewriting resume...
    Done.

  CONSENSUS AGENT — QUANTITATIVE EVALUATION REPORT

  [test_01]  Priya Nair — HR Manager  →  Target: HR Manager
------------------------------------------------------------------------
  Tool 1 — Resume parser                   F1=0.33  (P=0.33  R=0.33  6 skills  2.19s)
  Tool 2 — Job requirements fetch          F1=0.00  (P=0.00  R=0.00  src=llm_fallback  0.77s)
  Tool 3 — Gap (missing ski

Download Results

In [28]:
from google.colab import files
files.download('/content/consensus/eval_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
!zip -r concensus.zip /content/consensus/

  adding: content/consensus/ (stored 0%)
  adding: content/consensus/eval.py (deflated 72%)
  adding: content/consensus/__pycache__/ (stored 0%)
  adding: content/consensus/__pycache__/tools.cpython-312.pyc (deflated 49%)
  adding: content/consensus/__pycache__/agent.cpython-312.pyc (deflated 46%)
  adding: content/consensus/__pycache__/eval.cpython-312.pyc (deflated 49%)
  adding: content/consensus/agent.py (deflated 72%)
  adding: content/consensus/eval_data/ (stored 0%)
  adding: content/consensus/eval_data/resume_01.pdf (deflated 32%)
  adding: content/consensus/eval_data/.ipynb_checkpoints/ (stored 0%)
  adding: content/consensus/eval_data/resume_02.pdf (deflated 8%)
  adding: content/consensus/eval_data/resume_03.pdf (deflated 14%)
  adding: content/consensus/eval_results.json (deflated 74%)
  adding: content/consensus/app.py (deflated 71%)
  adding: content/consensus/tools.py (deflated 71%)


In [33]:
from google.colab import files
files.download('concensus.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>